# 04 - Model Explainability using SHAP

### Objective:
Explain model predictions globally (feature importances) and locally (individual customer waterfall plots) using SHAP values.


In [ ]:
import sys
import os
import importlib
sys.path.append("..")

import utils.data_loader
import utils.preprocessing

importlib.reload(utils.data_loader)
importlib.reload(utils.preprocessing)

import joblib
import shap
import pandas as pd
from utils.data_loader import load_raw_dataset
from utils.feature_engineering import add_engineered_features


In [ ]:
# Load model artifact
artifact = joblib.load("../models/best_churn_model.joblib")
model = artifact["model"]
preprocessor = artifact["preprocessor"]
feature_names = artifact["feature_names"]
num_cols = artifact.get("num_cols", [])
cat_cols = artifact.get("cat_cols", [])

raw_df = load_raw_dataset("../Telco_customer_churn.xlsx")
df_engineered = add_engineered_features(raw_df)

X = df_engineered.drop(columns=["customer_id", "churn"]).copy()

# Ensure exact dtype matching with preprocessor
for col in cat_cols:
    if col in X.columns:
        X[col] = X[col].astype(str)
for col in num_cols:
    if col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce").fillna(0.0).astype(float)

X_proc = preprocessor.transform(X)

# Compute SHAP values using TreeExplainer
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_proc[:500])

print("SHAP values computed for top 500 samples.")
shap.summary_plot(shap_values, X_proc[:500], feature_names=feature_names)
